In [1]:
import pandas as pd
from faker import Faker
import numpy as np
import random
from datetime import datetime, timedelta, date
import json

In [2]:
# Initialize Faker
fake = Faker()

In [3]:
# Generate users dataframe
num_users = 100
users_data = []

for i in range(num_users):
    users_data.append({
        'user_id': i + 1,
        'name': fake.name(),
        'email': fake.email(),
        'phone': fake.phone_number(),
        'address': fake.address(),
        'joined_at': fake.date_time_between_dates(date(2025, 1, 1), date(2026, 1, 10))
    })

users_df = pd.DataFrame(users_data)

print(f"\nTotal users: {len(users_df)}")


Total users: 100


In [4]:
# Generate prescriptions dataframe
num_prescriptions = 500
shipping_partners = ['FedEx', 'Purolator', 'Canada Post', 'DHL', 'MWS']
delivery_statuses = ['Pending', 'Processed', 'In Transit', 'Delivered', 'Cancelled']

prescriptions_data = []

for i in range(num_prescriptions):
    
    sub_total = round(random.uniform(20, 500), 2)
    insured_amount = round(sub_total * random.uniform(0.1, 0.3), 2)
    shipping_fee = round(random.uniform(5, 50), 2)
    total_amount = sub_total + shipping_fee
    
    prescriptions_data.append({
        'prescription_id': i + 1,
        'user_id': random.randint(1, num_users),
        'shipping_fee': shipping_fee,
        'sub_total': sub_total,
        'insured_amount': insured_amount,
        'total_amount': total_amount,
        'shipping_partner': random.choice(shipping_partners),
        'delivery_status': random.choice(delivery_statuses)
    })

prescriptions_df = pd.DataFrame(prescriptions_data)
print(f"\nTotal prescriptions: {len(prescriptions_df)}")


Total prescriptions: 500


In [5]:
prescriptions_df = prescriptions_df.merge(users_df, on='user_id', how='left')

In [6]:
prescriptions_df["order_date"] = prescriptions_df["joined_at"] + pd.to_timedelta(np.random.randint(0, 30, size=len(prescriptions_df)), unit='d')
prescriptions_df["delivery_date"] = prescriptions_df["order_date"] + pd.to_timedelta(np.random.randint(0, 8, size=len(prescriptions_df)), unit='d')

In [7]:
prescriptions_df.loc[prescriptions_df['delivery_status'].isin(['Pending', 'Processed', 'In Transit', 'Cancelled']), ['delivery_date']] = pd.NaT

In [8]:
prescriptions_df.loc[prescriptions_df['delivery_status'] == 'Cancelled', ['sub_total', 'total_amount', 'insured_amount']] = 0

In [9]:
prescriptions_df = prescriptions_df[['prescription_id', 'user_id', 'shipping_fee', 'sub_total',
       'insured_amount', 'total_amount', 'shipping_partner', 'delivery_status','order_date', 'delivery_date']]

In [10]:
# Generate medications dataframe
# Drug reference data
drug_data = {
    'Amoxicillin': {'strength': '500 mg', 'form': 'Capsule', 'din': '00012345'},
    'Ibuprofen': {'strength': '200 mg', 'form': 'Tablet', 'din': '00023456'},
    'Metformin': {'strength': '500 mg', 'form': 'Tablet', 'din': '00034567'},
    'Lisinopril': {'strength': '10 mg', 'form': 'Tablet', 'din': '00045678'},
    'Atorvastatin': {'strength': '20 mg', 'form': 'Tablet', 'din': '00056789'},
    'Omeprazole': {'strength': '20 mg', 'form': 'Capsule', 'din': '00067890'},
    'Aspirin': {'strength': '81 mg', 'form': 'Tablet (chewable)', 'din': '00078901'},
    'Salbutamol': {'strength': '100 mcg', 'form': 'Inhaler', 'din': '00089012'},
    'Sertraline': {'strength': '50 mg', 'form': 'Tablet', 'din': '00090123'},
    'Levothyroxine': {'strength': '75 mcg', 'form': 'Tablet', 'din': '00101234'}
}

medication_statuses = ['filled', 'rejected']
medications_data = []
medication_id = 1

for prescription_id, prescription_row in prescriptions_df.iterrows():
    
    num_meds = random.randint(1, 3)
    
    for med_idx in range(num_meds):
        # Select a random medication from the drug data
        medication_name = random.choice(list(drug_data.keys()))
        med_info = drug_data[medication_name]
        
        medications_data.append({
            'medication_id': medication_id,
            'prescription_id': prescription_id + 1,  # prescription_id is 1-indexed
            'medication_name': medication_name,
            'sku': fake.bothify(text='???-####'),
            'din': med_info['din'],
            'strength': med_info['strength'],
            'form': med_info['form'],
        })
        
        medication_id += 1

medications_df = pd.DataFrame(medications_data)

print(f"\nTotal medications: {len(medications_df)}")


Total medications: 1008


In [12]:
medications_df = medications_df.merge(prescriptions_df[["prescription_id", "delivery_status"]], on="prescription_id")

In [13]:
medications_df["delivery_status"].unique()

array(['Delivered', 'In Transit', 'Pending', 'Processed', 'Cancelled'],
      dtype=object)

In [16]:
medications_df["medication_status"] = "filled"
medications_df.loc[medications_df["delivery_status"] == "Cancelled", "medication_status"] = "rejected"

In [17]:
rejection_reasons = ["invalid_prescription", "out_of_stock", "insurance_denied", "patient_no_eligible", "controlled_substance"]
for i, row in medications_df.iterrows():
    insured_amount = prescription_row['insured_amount']
    status = row["medication_status"]
    rejection_reason = None if status == 'filled' else random.choice(rejection_reasons)
    # Generate plans where sum equals insured_amount
    num_plans = random.randint(1, 3)

    if status == 'filled':
                        # Create plans that sum to insured_amount
        if num_plans == 1:
            plans = [{'plan_name': f'Plan_{fake.random_letter()}', 'amount': round(insured_amount, 2)}]
        else:
            # Distribute insured_amount across multiple plans
            amounts = []
            remaining = insured_amount
            
            for i in range(num_plans - 1):
                amount = round(remaining / (num_plans - i               ) * random.uniform(0.3, 0.7), 2)
                amounts.append(amount)
                remaining -= amount
            
            amounts.append(round(remaining, 2))
            
            plans = [{'plan_name': f'Plan_{fake.random_letter()}', 'amount': amount} 
                    for amount in amounts]
    else:
        # For rejected medications, still create plan structure (null or minimal)
        plans = [{'plan_name': 'N/A', 'amount': 0}] 
    medications_df.at[i, "plans"] = json.dumps(plans)
    medications_df.at[i, "rejection_reason"] = rejection_reason

In [ ]:
# users_df.to_json("users_data.json")
# prescriptions_df.to_json("prescriptions_data.json")
# medications_df.to_json("medications_data.json")

In [18]:
users_df.to_csv("users_data.csv", index=False)
prescriptions_df.to_csv("prescriptions_data.csv", index=False)
medications_df.to_csv("medications_data.csv", index=False)